# Import Library

In [1]:
import kagglehub
import pandas as pd

/Users/ani/Projects/5_recommendation_system/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Download Data

In [ ]:
# # Download latest version
# path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")

# print("Path to dataset files:", path)

# Merge Data

In [112]:
df_credits = pd.read_csv("/Users/ani/Projects/5_recommendation_system/data/tmdb_5000_credits.csv").rename(columns={'movie_id': 'id'}).drop('title', axis=1)
df_credits.head(2)

,id,cast,crew
0,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [189]:
df_movies = pd.read_csv("/Users/ani/Projects/5_recommendation_system/data/tmdb_5000_movies.csv").drop(['original_title', 'homepage', 'status'], axis=1)
df_movies.head(2)

,budget,genres,id,keywords,original_language,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]","At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [190]:
df_merge = df_movies.merge(df_credits, on='id')

assert len(df_merge) == len(df_movies)
assert len(df_merge) == len(df_credits)

df_merge.head(2)

,budget,genres,id,keywords,original_language,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,tagline,title,vote_average,vote_count,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Enter the World of Pandora.,Avatar,7.2,11800,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]","At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [191]:
df_merge.columns.tolist()

['budget',
 'genres',
 'id',
 'keywords',
 'original_language',
 'overview',
 'popularity',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'spoken_languages',
 'tagline',
 'title',
 'vote_average',
 'vote_count',
 'cast',
 'crew']

# Pre-Process Data

#### Extract names from fields with JSON-like structures

In [192]:
# Genres
df_merge['genres'] = df_merge['genres'].apply(lambda x: eval(x) if isinstance(x, str) else x)
df_merge['genres'] = df_merge['genres'].apply(lambda x: [genre['name'] for genre in x] if isinstance(x, list) else 'N/A')
df_merge['genres'] = df_merge['genres'].apply(lambda x: ','.join(x) if isinstance(x, list) else x)

# characters
df_merge['characters'] = df_merge['cast'].apply(lambda x: eval(x) if isinstance(x, str) else x)
df_merge['characters'] = df_merge['characters'].apply(lambda x: [genre['character'] for genre in x] if isinstance(x, list) else 'N/A')
df_merge['characters'] = df_merge['characters'].apply(lambda x: ','.join(x) if isinstance(x, list) else x)

# actor
df_merge['actor'] = df_merge['cast'].apply(lambda x: eval(x) if isinstance(x, str) else x)
df_merge['actor'] = df_merge['actor'].apply(lambda x: [genre['name'] for genre in x] if isinstance(x, list) else 'N/A')
df_merge['actor'] = df_merge['actor'].apply(lambda x: ','.join(x) if isinstance(x, list) else x)

# director
df_merge['director'] = df_merge['crew'].apply(lambda x: next((i['name'] for i in eval(x) if i['job'] == 'Director'), None))

# production_companies
df_merge['production_companies'] = df_merge['production_companies'].apply(lambda x: eval(x) if isinstance(x, str) else x)
df_merge['production_companies'] = df_merge['production_companies'].apply(lambda x: [genre['name'] for genre in x] if isinstance(x, list) else 'N/A')
df_merge['production_companies'] = df_merge['production_companies'].apply(lambda x: ','.join(x) if isinstance(x, list) else x)

# production_countries
df_merge['production_countries'] = df_merge['production_countries'].apply(lambda x: eval(x) if isinstance(x, str) else x)
df_merge['production_countries'] = df_merge['production_countries'].apply(lambda x: [genre['name'] for genre in x] if isinstance(x, list) else 'N/A')
df_merge['production_countries'] = df_merge['production_countries'].apply(lambda x: ','.join(x) if isinstance(x, list) else x)

# spoken_languages
df_merge['spoken_languages'] = df_merge['spoken_languages'].apply(lambda x: eval(x) if isinstance(x, str) else x)
df_merge['spoken_languages'] = df_merge['spoken_languages'].apply(lambda x: [genre['name'] for genre in x] if isinstance(x, list) else 'N/A')
df_merge['spoken_languages'] = df_merge['spoken_languages'].apply(lambda x: ','.join(x) if isinstance(x, list) else x)

# keywords
df_merge['keywords'] = df_merge['keywords'].apply(lambda x: eval(x) if isinstance(x, str) else x)
df_merge['keywords'] = df_merge['keywords'].apply(lambda x: [genre['name'] for genre in x] if isinstance(x, list) else 'N/A')
df_merge['keywords'] = df_merge['keywords'].apply(lambda x: ','.join(x) if isinstance(x, list) else x)

df_merge = df_merge.drop(['cast', 'crew'], axis=1)

df_merge

,budget,genres,id,keywords,original_language,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,tagline,title,vote_average,vote_count,characters,actor,director
0,237000000,"Action,Adventure,Fantasy,Science Fiction",19995,"culture clash,future,space war,space colony,so...",en,"In the 22nd century, a paraplegic Marine is di...",150.437577,"Ingenious Film Partners,Twentieth Century Fox ...","United States of America,United Kingdom",2009-12-10,2787965087,162.0,"English,Español",Enter the World of Pandora.,Avatar,7.2,11800,"Jake Sully,Neytiri,Dr. Grace Augustine,Col. Qu...","Sam Worthington,Zoe Saldana,Sigourney Weaver,S...",James Cameron
1,300000000,"Adventure,Fantasy,Action",285,"ocean,drug abuse,exotic island,east india trad...",en,"Captain Barbossa, long believed to be dead, ha...",139.082615,"Walt Disney Pictures,Jerry Bruckheimer Films,S...",United States of America,2007-05-19,961000000,169.0,English,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,"Captain Jack Sparrow,Will Turner,Elizabeth Swa...","Johnny Depp,Orlando Bloom,Keira Knightley,Stel...",Gore Verbinski
2,245000000,"Action,Adventure,Crime",206647,"spy,based on novel,secret agent,sequel,mi6,bri...",en,A cryptic message from Bond’s past sends him o...,107.376788,"Columbia Pictures,Danjaq,B24","United Kingdom,United States of America",2015-10-26,880674609,148.0,"Français,English,Español,Italiano,Deutsch",A Plan No One Escapes,Spectre,6.3,4466,"James Bond,Blofeld,Madeleine,M,Lucia,Q,Moneype...","Daniel Craig,Christoph Waltz,Léa Seydoux,Ralph...",Sam Mendes
3,250000000,"Action,Crime,Drama,Thriller",49026,"dc comics,crime fighter,terrorist,secret ident...",en,Following the death of District Attorney Harve...,112.312950,"Legendary Pictures,Warner Bros.,DC Entertainme...",United States of America,2012-07-16,1084939099,165.0,English,The Legend Ends,The Dark Knight Rises,7.6,9106,"Bruce Wayne / Batman,Alfred Pennyworth,James G...","Christian Bale,Michael Caine,Gary Oldman,Anne ...",Christopher Nolan
4,260000000,"Action,Adventure,Science Fiction",49529,"based on novel,mars,medallion,space travel,pri...",en,"John Carter is a war-weary, former military ca...",43.926995,Walt Disney Pictures,United States of America,2012-03-07,284139100,132.0,English,"Lost in our world, found in another.",John Carter,6.1,2124,"John Carter,Dejah Thoris,Sola,Tars Tarkas,Tal ...","Taylor Kitsch,Lynn Collins,Samantha Morton,Wil...",Andrew Stanton
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4798,220000,"Action,Crime,Thriller",9367,"united states–mexico barrier,legs,arms,paper k...",es,El Mariachi just wants to play his guitar and ...,14.269792,Columbia Pictures,"Mexico,United States of America",1992-09-04,2040920,81.0,Español,"He didn't come looking for trouble, but troubl...",El Mariachi,6.6,238,"El Mariachi,Bigotón,Mauricio (Moco),Azul,Canti...","Carlos Gallardo,Jaime de Hoyos,Peter Marquardt...",Robert Rodriguez
4799,9000,"Comedy,Romance",72766,,en,A newlywed couple's honeymoon is upended by th...,0.642552,,,2011-12-26,0,85.0,,A newlywed couple's honeymoon is upended by th...,Newlyweds,5.9,5,"Buzzy,Linda,Marsha,Katie,Vanessa","Edward Burns,Kerry Bishé,Marsha Dietlein,Caitl...",Edward Burns
4800,0,"Comedy,Drama,Romance,TV Movie",231617,"date,love at first sight,narration,investigati...",en,"""Signed, Sealed, Delivered"" introduces a dedic...",1.444476,"Front Street Pictures,Muse Entertainment Enter...",United States of America,2013-10-13,0,120.0,English,NaN,"Signed, Sealed, Delivered",7.0,6,"Oliver O’Toole,Shane McInerney,Rita Haywith,No...","Eric Mabius,Kristin Booth,Crystal Lowe,Geoff G...",Scott Smith
4801,0,,126186,,en,When ambitious New York attorney Sam is sent t...,0.857008,,"United States of America,China",2012-05-03,0,98.0,English,A New Yorker in Shanghai,Shanghai Calling,5.7,7,"Sam,Amanda,Donald,Marcus Groff,","Daniel Henney,Eliza Coupe,Bill Pa

#### Re-order Field Names

In [196]:
all_columns = df_merge.columns.tolist()
primary_key = ['id']
movie_description_columns = ['title', 'overview', 'tagline', 'genres', 'release_date', 'runtime']
credits_columns = ['characters', 'actor', 'director', 'production_companies', 'production_countries']
ratings_columns = ['vote_average', 'vote_count', 'popularity']
other_columns = [col for col in all_columns if col not in primary_key + movie_description_columns + credits_columns + ratings_columns]

In [197]:
df_merge = df_merge[primary_key + movie_description_columns + ratings_columns + credits_columns + other_columns]
df_merge.head(2)

,id,title,overview,tagline,genres,release_date,runtime,vote_average,vote_count,popularity,characters,actor,director,production_companies,production_countries,budget,keywords,original_language,revenue,spoken_languages
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,"Action,Adventure,Fantasy,Science Fiction",2009-12-10,162.0,7.2,11800,150.437577,"Jake Sully,Neytiri,Dr. Grace Augustine,Col. Qu...","Sam Worthington,Zoe Saldana,Sigourney Weaver,S...",James Cameron,"Ingenious Film Partners,Twentieth Century Fox ...","United States of America,United Kingdom",237000000,"culture clash,future,space war,space colony,so...",en,2787965087,"English,Español"
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","At the end of the world, the adventure begins.","Adventure,Fantasy,Action",2007-05-19,169.0,6.9,4500,139.082615,"Captain Jack Sparrow,Will Turner,Elizabeth Swa...","Johnny Depp,Orlando Bloom,Keira Knightley,Stel...",Gore Verbinski,"Walt Disney Pictures,Jerry Bruckheimer Films,S...",United States of America,300000000,"ocean,drug abuse,exotic island,east india trad...",en,961000000,English


In [198]:
df_merge.columns.tolist()

['id',
 'title',
 'overview',
 'tagline',
 'genres',
 'release_date',
 'runtime',
 'vote_average',
 'vote_count',
 'popularity',
 'characters',
 'actor',
 'director',
 'production_companies',
 'production_countries',
 'budget',
 'keywords',
 'original_language',
 'revenue',
 'spoken_languages']

# Save Output

In [ ]:
df_merge.to_csv("/Users/ani/Projects/5_recommendation_system/data/df_movies_processed.csv", index=False)